# LUNG-CANC'AIR — Notebook principal

> **Principe :** Ce notebook contient uniquement les chemins, la configuration et des appels d'une ligne.
> Tout le code analytique est dans `lungcancair_analyses.py`.

**Workflow :**
1. Modifier `CHEMINS` (Partie 0) → chemins vers tes fichiers
2. Modifier `CFG` (Partie 1) → tes paramètres d'analyse
3. Exécuter les parties 2 à 9 dans l'ordre


---
## PARTIE 0 — Chemins des fichiers

> Modifier ici les chemins vers tes données.

In [1]:
# =============================================================================
# CHEMINS — Modifier ici
# =============================================================================

CHEMINS = {
    # Données de pollution journalière (INERIS)
    'pollution' : r"R:\Direction_Data\0_Projets\Projet_CANCAIR\pneumodetect\donnees patients\pneumodetect_cohorte_idf_ineris_data.csv",

    # Données cliniques (patients geocodés)
    'clinique'  : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_geocoded_clean_idf_2018_2023.csv",

    # Indice de défaveur sociale (EDI2021)
    'edi'       : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\edi2021-iris-fm.xlsx",

    # Shapefile routes nationales (TMJA)
    'tmja'      : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\tmja-rrnc-2024-shp",

    # Fichier ICPE (shapefile)
    'icpe'      : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\industries_a_risques\icpe.geojson\icpe_idf.shp",
}

print("✅ Chemins définis")


✅ Chemins définis


---
## PARTIE 1 — ★ CONFIGURATION ★

> **C'est la seule cellule à modifier entre deux analyses.**
> Toutes les analyses des parties suivantes héritent automatiquement de `CFG`.


In [2]:
# =============================================================================
# ★ CONFIGURATION CENTRALE ★
# =============================================================================

CFG = {

    # ── Seuils pour les variables % du temps (calcul Partie 2) ───────────────
    # Ces seuils définissent les colonnes créées dans df_final
    'seuils_pct' : {
        'PM25': [5, 10, 15, 25, 35],
        'PM10': [35, 45, 50, 60, 80, 90],
        'O3'  : [100, 120, 180],
    },

    # ── Paramètres routiers ───────────────────────────────────────────────────
    'shapefile_tmja' : CHEMINS['tmja'],   # chemin hérité de CHEMINS
    'buffer_rn_m'    : 500,               # ← rayon de proximité route (m)

    # ── Paramètres ICPE ───────────────────────────────────────────────────────
    'icpe_path'     : CHEMINS['icpe'],    # chemin hérité de CHEMINS
    'rayons_icpe_m' : [3000, 5000],       # ← rayons en mètres (ajouter/retirer librement)
    'icpe_types'    : {                   # ← types à comptabiliser séparément
        'NS': 'NS',                       # Non-Seveso
        'SB': 'SB',                       # Seveso Seuil Bas
        'SH': 'SH',                       # Seveso Seuil Haut
    },                                    # ← mettre {} pour total uniquement

    # ── Variables pour les régressions ───────────────────────────────────────
    # Polluants — cumul
    'cumul'    : ['PM25_cumul_120m', 'PM10_cumul_120m', 'O3_cumul_120m'],

    # Polluants — tendance Theil-Sen
    'tendance' : ['PM25_mm365_sen_pente', 'PM10_mm365_sen_pente', 'O3_mm365_sen_pente'],

    # % du temps — seuils retenus après Partie 4
    'pct_pm25' : [10],         # → PM25_pct_sup10
    'pct_pm10' : [35],         # → PM10_pct_sup35
    'pct_o3'   : [100, 120],   # → O3_pct_sup100, O3_pct_sup120

    # Variables ICPE dans les modèles
    # Choisir parmi les colonnes créées (nb_ICPE_total_3km, nb_ICPE_SH_3km,
    #   nb_ICPE_SB_5km, dist_ICPE_SH_m, dist_ICPE_SB_m, ...)
    'icpe'     : ['nb_ICPE_total_3km', 'dist_ICPE_SH_m'],

    # Variables cliniques
    'inclure_age'    : True,
    'inclure_paquet' : True,   # False automatique dans C vs D et A+C vs B+D
    'inclure_sexe'   : True,
    'inclure_edi'    : True,
    'inclure_trafic' : True,

    # ── Filtres population (None = pas de filtre) ─────────────────────────────
    'filtre_sexe'       : None,   # None | 'feminin' | 'masculin'
    'filtre_fumeur'     : None,   # None | True | False
    'filtre_histologie' : None,   # None | 'adenocarcinome' | 'epidermoide'
    'filtre_custom'     : {},     # ex: {'annee_diagnostic': 2020}

    # ── Dose-réponse ──────────────────────────────────────────────────────────
    'polluant_dr_AB'  : 'PM25_cumul_120m',   # polluant dose-réponse A vs B
    'polluant_dr_CD'  : 'O3_cumul_120m',     # polluant dose-réponse C vs D
    'n_quintiles'     : 5,

    # ── Dose-réponse spatiale ─────────────────────────────────────────────────
    'col_distance' : 'dist_ICPE_SH_m',       # variable de distance
    'tranches_km'  : [0, 1, 3, 5, 10, 999],  # tranches en km
}

print("✅ Configuration définie")


✅ Configuration définie


---
## PARTIE 2 — Construction de df_final

> Exécuter **une seule fois** au démarrage. Ne pas ré-exécuter entre deux analyses.

In [4]:
import sys
sys.path.append('src')
import importlib
import lungcancair_analyses as lca
importlib.reload(lca)

# ── Chargement des données brutes ─────────────────────────────────────────────
data, df_clinique = lca.charger_donnees(CHEMINS)

# ── Calcul des variables de pollution (10 ans) ────────────────────────────────
df_air = lca.calculer_variables_air(data, df_clinique, CFG)

# ── Variables socio-économiques ───────────────────────────────────────────────
df_air = lca.ajouter_socioeco(df_air, df_clinique, CHEMINS['edi'])

# ── Variables routières ───────────────────────────────────────────────────────
df_air, gdf_patients = lca.ajouter_routier(df_air, df_clinique, CFG)

# ── Variables ICPE ────────────────────────────────────────────────────────────
df_air = lca.ajouter_icpe(df_air, gdf_patients, CFG)

# ── Fusion finale → df_final ──────────────────────────────────────────────────
df_final = lca.construire_df_final(data, df_clinique, df_air)

print(f"\n✅ df_final prêt : {len(df_final)} patients × {len(df_final.columns)} variables")
print(f"   Colonnes ICPE disponibles : {[c for c in df_final.columns if 'ICPE' in c]}")


✅ data        : 2,993 patients
✅ df_clinique : 1,682 patients
Patients communs : 1682


Patients:  39%|███▉      | 658/1682 [04:13<06:34,  2.59it/s]


KeyboardInterrupt: 

#### 📊 Résultats — Construction df_final

*À compléter après exécution.*

---
## PARTIE 3 — Exploration visuelle

In [ ]:
lca.explorer_distribution_polluants(df_final)

#### 📊 Résultats — Distribution polluants

*À compléter après exécution.*

In [ ]:
lca.explorer_distribution_pct(df_final)

#### 📊 Résultats — Distribution % du temps

*À compléter après exécution.*

In [ ]:
lca.explorer_tendances(df_final)

#### 📊 Résultats — Tendances MM365+MK

*À compléter après exécution.*

---
## PARTIE 4 — Seuils critiques d'exposition

In [ ]:
df_res_seuils = lca.chercher_seuils_critiques(df_final, CFG)

#### 📊 Résultats — Seuils critiques

*À compléter après exécution.*

---
## PARTIE 5 — Corrélation & FAMD

In [ ]:
# Heatmap 1 — Variables qualité de l'air
vars_heatmap1 = {
    'Qualité de l\'air — Moyenne' : ['PM25_moyenne','PM10_moyenne','NO2_moyenne','O3_moyenne'],
    'Qualité de l\'air — Médiane' : ['PM25_mediane','PM10_mediane','NO2_mediane','O3_mediane'],
    'Qualité de l\'air — Cumul'   : ['PM25_cumul_120m','PM10_cumul_120m','NO2_cumul_120m','O3_cumul_120m'],
}
lca.heatmap_correlation(df_final, vars_heatmap1)


#### 📊 Résultats — Heatmap 1

*À compléter après exécution.*

In [ ]:
# Heatmap 2 — Variables retenues pour les modèles
vars_heatmap2 = {
    'Qualité de l\'air — Cumul' : CFG['cumul'],
    'Clinique'                   : ['age_diagnostic','paquet_annee','sexe_bin'],
    'Socio-économique'           : ['quintileEDI2021'],
    'Routier'                    : ['indice_trafic'],
    'Industriel'                 : CFG['icpe'],
}
lca.heatmap_correlation(df_final, vars_heatmap2)


#### 📊 Résultats — Heatmap 2

*À compléter après exécution.*

In [ ]:
# FAMD
for mut in ['EGFR','ALK','MET','ROS1','ERBB2','KRAS','BRAF']:
    df_final[f'mut_{mut}_bin'] = lca.porte_mutation(df_final, mut).map({True:'Positif',False:'Négatif'})

vars_famd_cont = (CFG['cumul'] + CFG['tendance'] +
    [f'PM25_pct_sup{s}' for s in CFG['pct_pm25']] +
    [f'PM10_pct_sup{s}' for s in CFG['pct_pm10']] +
    [f'O3_pct_sup{s}'   for s in CFG['pct_o3']]   +
    ['age_diagnostic','paquet_annee','quintileEDI2021','indice_trafic'] +
    CFG['icpe'])

vars_famd_cat = ['sexe','histologie_groupe','PM25_mm365_mk_tendance','O3_mm365_mk_tendance',
    'groupe_AB','groupe_CD','groupe_AC_BD',
    'mut_EGFR_bin','mut_ALK_bin','mut_MET_bin','mut_KRAS_bin','mut_BRAF_bin']

famd, coords, var_exp, df_famd = lca.run_famd(df_final, vars_famd_cont, vars_famd_cat)


#### 📊 Résultats — FAMD

*À compléter après exécution.*

---
## PARTIE 6 — Tableaux de caractéristiques

In [ ]:
t1, t2, t3 = lca.tableaux_caracteristiques(df_final)

#### 📊 Résultats — Tableaux de caractéristiques

*À compléter après exécution.*

---
## PARTIE 7 — Régressions logistiques principales

In [ ]:
resultats = lca.analyses_principales(df_final, CFG)

#### 📊 Résultats — Régressions principales

*À compléter après exécution.*

---
## PARTIE 8 — Mutations individuelles

| Mutation | N+  | EPV  | Fiabilité       |
|----------|-----|------|-----------------|
| EGFR     | 178 | 10.5 | ✅ Fiable        |
| MET      | 58  | 3.4  | ⚠️ Prudence      |
| ALK      | 54  | 3.2  | ⚠️ Prudence      |
| ERBB2    | 32  | 1.9  | ❌ Sous-puissant |
| ROS1     | 30  | 1.8  | ❌ Sous-puissant |


In [ ]:
resultats_mutations = lca.analyses_mutations(df_final, CFG, mutations=['EGFR','MET','ALK','ERBB2','ROS1'])

#### 📊 Résultats — Mutations individuelles

*À compléter après exécution.*

---
## PARTIE 9 — Analyses de sensibilité

In [ ]:
# Stratification par sexe
res_sexe_AB   = lca.analyse_stratifiee_sexe(df_final, CFG, groupe='AB')
res_sexe_CD   = lca.analyse_stratifiee_sexe(df_final, CFG, groupe='CD')
res_sexe_EGFR = lca.analyse_stratifiee_sexe(df_final, {**CFG, 'filtre_fumeur': None}, groupe='AB')


#### 📊 Résultats — Stratification sexe

*À compléter après exécution.*

In [ ]:
# Non-fumeurs uniquement
res_nonfumeurs = lca.analyse_sous_groupe(df_final, CFG,
    filtre_fumeur=False, groupe='AB',
    titre_custom='A vs B — Non-fumeurs uniquement')


#### 📊 Résultats — Non-fumeurs uniquement

*À compléter après exécution.*

In [ ]:
# Adénocarcinomes uniquement
res_adeno = lca.analyse_sous_groupe(df_final, CFG,
    filtre_histologie='adenocarcinome', groupe='AB',
    titre_custom='A vs B — Adénocarcinomes uniquement')


#### 📊 Résultats — Adénocarcinomes uniquement

*À compléter après exécution.*

---
## PARTIE 10 — Analyses dose-réponse

In [ ]:
# Dose-réponse par quintile — A vs B
dr_AB = lca.dose_reponse_quintiles(df_final, CFG,
    polluant    = CFG['polluant_dr_AB'],
    groupe      = 'AB',
    n_quintiles = CFG['n_quintiles'])


#### 📊 Résultats — Dose-réponse quintiles — A vs B

*À compléter après exécution.*

In [ ]:
# Dose-réponse par quintile — C vs D
dr_CD = lca.dose_reponse_quintiles(df_final, CFG,
    polluant    = CFG['polluant_dr_CD'],
    groupe      = 'CD',
    n_quintiles = CFG['n_quintiles'])


#### 📊 Résultats — Dose-réponse quintiles — C vs D

*À compléter après exécution.*

In [ ]:
# Dose-réponse spatiale EGFR × Distance ICPE
ds_egfr = lca.dose_reponse_spatiale(df_final, CFG,
    col_distance = CFG['col_distance'],
    mutation     = 'EGFR',
    tranches_km  = CFG['tranches_km'])


#### 📊 Résultats — Dose-réponse spatiale EGFR

*À compléter après exécution.*

In [ ]:
# Dose-réponse spatiale — A vs B
ds_AB = lca.dose_reponse_spatiale(df_final, CFG,
    col_distance = CFG['col_distance'],
    groupe       = 'AB',
    tranches_km  = CFG['tranches_km'])


#### 📊 Résultats — Dose-réponse spatiale — A vs B

*À compléter après exécution.*

---
## PARTIE 11 — Variables % du temps

In [ ]:
df_res_pct = lca.analyse_pct_temps(df_final, CFG)

#### 📊 Résultats — Variables % du temps

*À compléter après exécution.*

---
## PARTIE 12 — Synthèse comparative

In [ ]:
df_synthese = lca.synthese_comparaison({
    'A vs B'              : resultats.get('AB'),
    'C vs D'              : resultats.get('CD'),
    'A+C vs B+D'          : resultats.get('ACBD'),
    'EGFR ✅'             : resultats_mutations.get('EGFR'),
    'MET ⚠️'              : resultats_mutations.get('MET'),
    'ALK ⚠️'              : resultats_mutations.get('ALK'),
    'Non-fumeurs'         : res_nonfumeurs,
    'Adénocarcinomes'     : res_adeno,
    'A vs B — Femmes'     : res_sexe_AB.get('feminin'),
    'A vs B — Hommes'     : res_sexe_AB.get('masculin'),
})
